Bronze Layer - Binance Crypto Streaming

In [0]:
# Configuration using secrets
EVENT_HUB_NAMESPACE = "binance-eventshub"
EVENT_HUB_NAME = "binance-trades"
EVENT_HUB_ACCESS_KEY_NAME = "policy"

# Retrieve secrets securely
EVENT_HUB_ACCESS_KEY = dbutils.secrets.get(scope="binance-secrets", key="eventhub-access-key")
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope="binance-secrets", key="storage-account-key")

CONNECTION_STRING = f"Endpoint=sb://{EVENT_HUB_NAMESPACE}.servicebus.windows.net/;SharedAccessKeyName={EVENT_HUB_ACCESS_KEY_NAME};SharedAccessKey={EVENT_HUB_ACCESS_KEY};EntityPath={EVENT_HUB_NAME}"

storage_account = "deproj1adls"
container = "crypto-data"

spark.conf.set(f"fs.azure.account.key.{storage_account}.blob.core.windows.net", STORAGE_ACCOUNT_KEY)

CHECKPOINT_PATH = f"wasbs://{container}@{storage_account}.blob.core.windows.net/checkpoints/bronze_binance_v3"
BRONZE_TABLE_PATH = f"wasbs://{container}@{storage_account}.blob.core.windows.net/bronze/binance_crypto_raw_v3"
TRIGGER_INTERVAL = "10 seconds"

print("✓ Configuration loaded successfully using secrets")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

In [0]:
ehConf = {
    'eventhubs.connectionString': sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(CONNECTION_STRING),
    'eventhubs.consumerGroup': '$Default',
    'maxEventsPerTrigger': 1000
}

event_schema = StructType([
    StructField("source", StringType(), True),
    StructField("stream", StringType(), True),
    StructField("ingestion_timestamp", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", LongType(), True),
    StructField("symbol", StringType(), True),
    StructField("trade_data", StructType([
        StructField("event_type", StringType(), True),
        StructField("event_time_ms", LongType(), True),
        StructField("symbol", StringType(), True),
        StructField("trade_id", LongType(), True),
        StructField("price", StringType(), True),
        StructField("quantity", StringType(), True),
        StructField("buyer_order_id", LongType(), True),
        StructField("seller_order_id", LongType(), True),
        StructField("trade_time", LongType(), True),
        StructField("is_buyer_maker", BooleanType(), True),
        StructField("is_best_match", BooleanType(), True),
        StructField("price_change_percent", StringType(), True),
        StructField("close_price", StringType(), True),
        StructField("open_price", StringType(), True),
        StructField("high_price", StringType(), True),
        StructField("low_price", StringType(), True),
        StructField("volume", StringType(), True),
    ]), True)
])

In [0]:
raw_stream = (
    spark.readStream
    .format("eventhubs")
    .options(**ehConf)
    .load()
)

raw_stream.printSchema()

In [0]:
parsed_stream = (
    raw_stream
    .select(
        col("enqueuedTime").alias("eventhub_enqueued_time"),
        col("offset").alias("eventhub_offset"),
        col("sequenceNumber").alias("eventhub_sequence_number"),
        col("partitionKey").alias("eventhub_partition_key"),
        col("body").cast("string").alias("raw_body"),
        current_timestamp().alias("bronze_ingestion_time"),
        lit("eventhub").alias("bronze_source_system"),
        from_json(col("body").cast("string"), event_schema).alias("parsed")
    )
)

bronze_stream = (
    parsed_stream
    .withColumn("source", col("parsed.source"))
    .withColumn("stream", col("parsed.stream"))
    .withColumn("ingestion_timestamp", col("parsed.ingestion_timestamp"))
    .withColumn("event_type", col("parsed.event_type"))
    .withColumn("event_time", col("parsed.event_time"))
    .withColumn("symbol", col("parsed.symbol"))
    .withColumn("trade_event_type", col("parsed.trade_data.event_type"))
    .withColumn("trade_event_time_ms", col("parsed.trade_data.event_time_ms"))
    .withColumn("trade_symbol", col("parsed.trade_data.symbol"))
    .withColumn("trade_id", col("parsed.trade_data.trade_id"))
    .withColumn("trade_price", col("parsed.trade_data.price"))
    .withColumn("trade_quantity", col("parsed.trade_data.quantity"))
    .withColumn("buyer_order_id", col("parsed.trade_data.buyer_order_id"))
    .withColumn("seller_order_id", col("parsed.trade_data.seller_order_id"))
    .withColumn("trade_time", col("parsed.trade_data.trade_time"))
    .withColumn("is_buyer_maker", col("parsed.trade_data.is_buyer_maker"))
    .withColumn("is_best_match", col("parsed.trade_data.is_best_match"))
    .withColumn("price_change_percent", col("parsed.trade_data.price_change_percent"))
    .withColumn("close_price", col("parsed.trade_data.close_price"))
    .withColumn("open_price", col("parsed.trade_data.open_price"))
    .withColumn("high_price", col("parsed.trade_data.high_price"))
    .withColumn("low_price", col("parsed.trade_data.low_price"))
    .withColumn("volume", col("parsed.trade_data.volume"))
    .drop("parsed")
)

In [0]:
bronze_query = (
    bronze_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start(BRONZE_TABLE_PATH)
)

print(f"Stream started: {bronze_query.id}")

In [0]:
import time

print(f"Is Active: {bronze_query.isActive}")
print("\nWaiting for data...")

for i in range(200):
    time.sleep(2)
    status = bronze_query.status
    print(f"\nBatch {i+1}:")
    print(f"  Message: {status['message']}")
    if 'numInputRows' in status:
        print(f"  Input Rows: {status['numInputRows']}")
    print(f"  Is Data Available: {status['isDataAvailable']}")

In [0]:
bronze_df = spark.read.format("delta").load(BRONZE_TABLE_PATH)

quality_check = bronze_df.select(
    count("*").alias("total_records"),
    count(when(col("symbol").isNull(), 1)).alias("null_symbols"),
    count(when(col("event_time").isNull(), 1)).alias("null_event_times"),
    countDistinct("symbol").alias("distinct_symbols"),
    countDistinct("event_type").alias("distinct_event_types")
)

display(quality_check)

In [0]:
def show_active_streams():
    active_streams = spark.streams.active
    print(f"Active Streams: {len(active_streams)}")
    for stream in active_streams:
        print(f"  ID: {stream.id} | Active: {stream.isActive}")

show_active_streams()

In [0]:
bronze_query.stop()

In [0]:
%sql
SELECT 
    date_trunc('minute', bronze_ingestion_time) as minute,
    COUNT(*) as events,
    COUNT(DISTINCT symbol) as unique_symbols
FROM delta.`wasbs://crypto-data@deproj1adls.blob.core.windows.net/bronze/binance_crypto_raw_v3`
WHERE bronze_ingestion_time >= current_timestamp() - INTERVAL 1 HOUR
GROUP BY date_trunc('minute', bronze_ingestion_time)
ORDER BY minute DESC

In [0]:
%sql
SELECT 
    symbol,
    event_type,
    from_unixtime(event_time/1000) as event_timestamp,
    trade_price,
    trade_quantity,
    is_buyer_maker
FROM delta.`wasbs://crypto-data@deproj1adls.blob.core.windows.net/bronze/binance_crypto_raw_v3`
WHERE event_type = 'trade'
ORDER BY event_time DESC
LIMIT 10